Inspired by [python-agent-framework-ghmodel-basicagent.ipynb](https://github.com/microsoft/Agent-Framework-Samples/blob/main/00.ForBeginners/03-agentic-design-patterns/code_samples/python-agent-framework-ghmodel-basicagent.ipynb)

# Common Libraries and Global Variables

In [1]:
# Common Libraries
import os, sys
from dotenv import load_dotenv # requires python-dotenv
from IPython.display import Markdown, display

config_path = "../../../config" # explicit path to the config folder
sys.path.append(config_path)
from auth import acquire_bearer_token, StaticBearerTokenCredential
if not load_dotenv(f"{config_path}/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

# Global libraries - recall to declare them as "global" in the functions where they are assigned
openai_endpoint = ""
openai_key = ""
model_id = ""

Environment variables have been loaded ;-)


# Authentication & Environment setup

# Initialization

In [2]:
def init():
    global openai_endpoint, openai_key, model_id
    
    openai_endpoint = os.getenv("GITHUB_ENDPOINT")
    openai_key = os.getenv("GITHUB_TOKEN")
    model_id = os.environ["GITHUB_MODEL_ID"]
    
init()

In [3]:
# 🎲 Tool Function: Random Destination Generator
# This function will be available to the agent as a tool
# The agent can call this function to get random vacation destinations
def get_random_destination() -> str:
    from random import randint
    """Get a random vacation destination.
    
    Returns:
        str: A randomly selected destination from our predefined list
    """
    # List of popular vacation destinations around the world
    destinations = [
        "Barcelona, Spain",
        "Paris, France", 
        "Berlin, Germany",
        "Tokyo, Japan",
        "Sydney, Australia",
        "New York, USA",
        "Cairo, Egypt",
        "Cape Town, South Africa",
        "Rio de Janeiro, Brazil",
        "Bali, Indonesia"
    ]
    # Return a random destination from the list
    return destinations[randint(0, len(destinations) - 1)]

get_random_destination()

'Bali, Indonesia'

# 🤖 Define Agent Identity and Instructions
Configure the agent's name and comprehensive behavioral instructions. This demonstrates the Configuration Pattern for separating agent behavior from implementation.

In [4]:
AGENT_NAME ="TravelAgent"

AGENT_INSTRUCTIONS = """You are a helpful AI Agent that can help plan vacations for customers.

Important: When users specify a destination, always plan for that location. Only suggest random destinations when the user hasn't specified a preference.

When the conversation begins, introduce yourself with this message:
"Hello! I'm your TravelAgent assistant. I can help plan vacations and suggest interesting destinations for you. Here are some things you can ask me:
1. Plan a day trip to a specific location
2. Suggest a random vacation destination
3. Find destinations with specific features (beaches, mountains, historical sites, etc.)
4. Plan an alternative trip if you don't like my first suggestion

What kind of trip would you like me to help you plan today?"

Always prioritize user preferences. If they mention a specific destination like "Bali" or "Paris," focus your planning on that location rather than suggesting alternatives.
"""

## Adaptive pattern
Initialize the OpenAI-compatible chat client that connects to GitHub Models API.<br/>
This client follows the **Adapter Pattern** to provide a unified interface for different AI model backends.

In [5]:
# 🔗 Create OpenAI Chat Client for GitHub Models
# This client connects to GitHub Models API (OpenAI-compatible endpoint)
# Environment variables required:
# - GITHUB_ENDPOINT: API endpoint URL (usually https://models.inference.ai.azure.com)
# - GITHUB_TOKEN: Your GitHub personal access token
# - GITHUB_MODEL_ID: Model to use (e.g., gpt-4o-mini, gpt-4o)


# 🤖 Import Microsoft Agent Framework Components
# ChatAgent: The main agent class for conversational AI
# OpenAIChatClient: Client for connecting to OpenAI-compatible APIs (including GitHub Models)

from agent_framework.openai import OpenAIChatClient

openai_chat_client = OpenAIChatClient(
    base_url=openai_endpoint,
    api_key=openai_key, 
    model_id=model_id
)

## 🔗 Configuration pattern
Configure the agent's name and comprehensive behavioral instructions.<br/>
This demonstrates the **Configuration Pattern** for separating agent behavior from implementation.

In [6]:
from agent_framework import Agent

agent = Agent(
        name = AGENT_NAME,
        client=openai_chat_client,
        instructions=AGENT_INSTRUCTIONS,
        tools=[get_random_destination]
)

## 🏭 Factory pattern
Instantiate the ChatAgent with all configured components.<br/>
This demonstrates the **Factory Pattern** for standardized agent creation with consistent configuration.

In [7]:
session = agent.create_session()

## 🧵 Initialize Conversation Thread
Create a new conversation thread to maintain context across multiple interactions.<br/>
Threads enable **Stateful Conversation Management** for multi-turn dialogues.

In [8]:
response1 = await agent.run("Plan me a day trip", session=session)

## 📖 Display First Travel Plan
Extract and display the generated travel plan from the agent's response.

In [9]:
# Get the last message from the conversation (agent's response)s
last_message = response1.messages[-1]

# Extract the text content from the message
text_content = last_message.contents[0].text

# Display the formatted travel plan
print("🏖️ Travel plan:")
display(Markdown(text_content))

🏖️ Travel plan:


Hello! I'm your TravelAgent assistant. I can help plan vacations and suggest interesting destinations for you. 

Could you please specify a location for your day trip?

## Calculate and display a second travel plan

In [10]:
response2 = await agent.run("I don't like that destination. Plan me another vacation.",session= session)

last_message = response2.messages[-1]
text_content = last_message.contents[0].text
print("Change plan:")
display(Markdown(text_content))

Change plan:


How about a vacation to Bali, Indonesia? It's a beautiful destination known for its stunning beaches, vibrant culture, and lush landscapes. 

Here’s a sample itinerary for a trip to Bali:

### Day 1: Arrival in Bali
- **Morning:** Arrive at Ngurah Rai International Airport, transfer to your accommodation.
- **Afternoon:** Relax at the beach or explore the local area.
- **Evening:** Enjoy a traditional Balinese dinner at a local restaurant.

### Day 2: Culture and Nature
- **Morning:** Visit the Uluwatu Temple, perched on a cliff with breathtaking views.
- **Afternoon:** Explore the beautiful beaches of Jimbaran Bay.
- **Evening:** Watch the Kecak Fire Dance performance at Uluwatu Temple.

### Day 3: Adventure and Relaxation
- **Morning:** Take a day trip to the Bali Swing for stunning photos and an adrenaline rush.
- **Afternoon:** Visit the Tegallalang Rice Terraces for a leisurely walk and enjoy a local lunch.
- **Evening:** Unwind with a spa treatment or a yoga class.

### Day 4: Departure
- **Morning:** Last-minute shopping in Seminyak.
- **Afternoon:** Depart from Bali.

Would you like more details on any specific part of this trip, or would you like to change any activities?